# 第十三課 - 代理記憶


## 設定

此筆記本示範如何使用 **Microsoft Agent Framework** (MAF) 建立具備 <strong>持久記憶</strong> 的旅遊預訂代理。

您將了解不同類型的代理記憶體 — 工作記憶、短期記憶及長期記憶 — 如何影響代理在對話中保留及使用資料。

**先決條件：**
- 擁有一個 Microsoft Foundry 專案，並部署了聊天模型 (例如 `gpt-5-mini`)。
- 已用 Azure CLI 登入 — 在終端機執行 `az login`。
- `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 專案端點。
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署模型的名稱。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 代理記憶類型

AI 代理可以利用不同類型的記憶，每種都有其獨特的用途：

### 工作記憶
對話線程本身 — 單一會話中交換的訊息。代理可以回顧同一線程中較早的訊息以保持連貫性。在 MAF 中，這是透過 **`agent.create_session()`** 建立的，該方法會返回一個 `AgentSession`。

### 短期記憶
資訊在任務或會話期間持續存在，但不會被永久儲存。例如，代理可能在多輪規劃對話中累積事實，並用以產生最終行程。

### 長期記憶
偏好和事實會 <strong>跨會話</strong> 持續存在。回訪用戶不應該重複他們的飲食限制或旅遊風格。長期記憶一般由外部儲存支援 — 例如資料庫、檔案或向量索引 — 並透過工具呈現給代理。


## 使用會話的工作記憶

記憶最簡單的形式是對話會話。當你將相同的會話（透過 `agent.create_session()` 創建）傳遞給連續的 `agent.run()` 調用時，代理會看到整個對話歷史，並能回憶起較早的細節。

讓我們創建一個旅遊代理並展示工作記憶。


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

代理正確地回憶起預算，因為兩條訊息屬於同一個會話。這就是<strong>工作記憶</strong>——只存在於會話期間。

### 新線程會發生什麼事？

如果我們建立一個<strong>新</strong>的會話，代理便不會記得之前的對話：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 長期記憶模式

為了記住用戶在<strong>多個會話之間</strong>的偏好，我們需要一個存在於對話線程之外的持久存儲。代理通過<strong>工具</strong>訪問此存儲——這些是它可以調用來保存和檢索資訊的函數。

以下我們實現一個簡單的記憶體中偏好存儲（在生產環境中你會用資料庫或向量索引來支持它），並將其作為代理可使用的工具暴露出來。

### 架構
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 場景 1 — 初次使用者預訂周年紀念旅行

Sarah 第一次造訪。代理應該透過工具儲存她的偏好，並利用這些偏好推薦酒店。


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 情境 2 — Sarah 數星期後回來

Sarah 開始一個<strong>全新線程</strong>（模擬新會話）。工作記憶是空的，但長期偏好儲存庫仍有她的資料。代理應檢索並使用它來個人化推薦。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 摘要

在這課中，你學習了三種類型的代理記憶體，以及如何用 Microsoft Agent Framework 實作它們：

| 記憶體類型 | MAF 機制 | 存活期 |
|---|---|---|
| <strong>工作記憶</strong> | `agent.create_session()` | 單次對話 |
| <strong>短期記憶</strong> | 線程內累積的上下文 | 單一任務 / 會話 |
| <strong>長期記憶</strong> | 透過 `@tool` 函式存取的外部儲存 | 跨會話 |

### 主要心得
1. **`agent.create_session()`** 提供工作記憶 — 代理可在一個會話中看到完整對話歷史。
2. <strong>新會話會失去上下文</strong> — 沒有長期記憶，代理無法回想過往對話。
3. **`@tool` 函式** 彌補此差距 — 讓代理能儲存和取回持久儲存的信息。
4. <strong>個人化隨時間改善</strong> — 存越多偏好，代理的推薦越精準。

### 實務應用
- <strong>客戶服務</strong>：記住客戶歷史與偏好
- <strong>個人助理</strong>：維持數日或數週的上下文
- <strong>醫療保健</strong>：追蹤病人資訊與偏好
- <strong>電子商務</strong>：根據歷史提供個人化購物體驗

### 後續步驟
- 用資料庫或向量儲存（如 Azure AI Search）替換記憶內存字典
- 為時效性資訊新增記憶過期機制
- 建構具共享記憶的多代理系統
- 探索 Cognee 筆記本支援知識圖譜的記憶功能


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
